# **Problem Statement**

## Business Context

Renewable energy sources play an increasingly important role in the global energy mix, as the effort to reduce the environmental impact of energy production increases.

Out of all the renewable energy alternatives, wind energy is one of the most developed technologies worldwide. The U.S Department of Energy has put together a guide to achieving operational efficiency using predictive maintenance practices.

Predictive maintenance uses sensor information and analysis methods to measure and predict degradation and future component capability. The idea behind predictive maintenance is that failure patterns are predictable and if component failure can be predicted accurately and the component is replaced before it fails, the costs of operation and maintenance will be much lower.

The sensors fitted across different machines involved in the process of energy generation collect data related to various environmental factors (temperature, humidity, wind speed, etc.) and additional features related to various parts of the wind turbine (gearbox, tower, blades, break, etc.).

## Objective

“ReneWind” is a company working on improving the machinery/processes involved in the production of wind energy using machine learning and has collected data of generator failure of wind turbines using sensors. They have shared a ciphered version of the data, as the data collected through sensors is confidential (the type of data collected varies with companies). Data has 40 predictors, 20000 observations in the training set and 5000 in the test set.

The objective is to build various classification models, tune them, and find the best one that will help identify failures so that the generators could be repaired before failing/breaking to reduce the overall maintenance cost.
The nature of predictions made by the classification model will translate as follows:

- True positives (TP) are failures correctly predicted by the model. These will result in repairing costs.
- False negatives (FN) are real failures where there is no detection by the model. These will result in replacement costs.
- False positives (FP) are detections where there is no failure. These will result in inspection costs.

It is given that the cost of repairing a generator is much less than the cost of replacing it, and the cost of inspection is less than the cost of repair.

“1” in the target variables should be considered as “failure” and “0” represents “No failure”.

## Data Description

The data provided is a transformed version of the original data which was collected using sensors.

- Train.csv - To be used for training and tuning of models.
- Test.csv - To be used only for testing the performance of the final best model.

Both the datasets consist of 40 predictor variables and 1 target variable.

# **Importing necessary libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, f1_score,accuracy_score, recall_score, precision_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout,BatchNormalization
from tensorflow.keras import backend

import warnings
warnings.filterwarnings("ignore")

random_seed = 11

In [ ]:
# Tensorflow setup items
tf.keras.utils.set_random_seed(11)
tf.config.experimental.enable_op_determinism()

# **Loading the Data**

In [ ]:
df = pd.read_csv("Train.csv")
df_test = pd.read_csv("Test.csv")

In [ ]:
df.head()

In [ ]:
df_test.head()

# **Data Overview**

### Checking the Shape of the Data

In [ ]:
df.shape

In [ ]:
df_test.shape

### Basic Data Statistics

In [ ]:
df.describe().T

### Review Data Types and First Look at Missing Data

In [ ]:
df.info()

In [ ]:
df_test.info()

All of the X values are float64 and the y (Target) in an int64. Both the training and test sets have missing values.

I will also scale the data base on the amount of spread in the data.

### Check for Duplicate Values
There are no duplicate values.

In [ ]:
df.duplicated().sum()

In [ ]:
df_test.duplicated().sum()

### Missing Values
The training dataset as a total of 36 missing values, 18 missing values in two rows. In the test dataset there are 11 missing values. 

In [ ]:
df.isnull().sum()

In [ ]:
df_test.isnull().sum()

In [ ]:
round(df.isnull().sum() / df.isnull().count() * 100, 2)

#### Dataset Balance

In [ ]:
class_counts = df['Target'].value_counts()
print(f"Class Counts: {class_counts}")
print("")
print("The percentage of class 1 in the training dataset")
class_counts[1] / (class_counts[0] + class_counts[1])

In [ ]:
class_counts_test = df_test['Target'].value_counts()
print(f"Class Counts: {class_counts_test}")
class_counts_test[1] / (class_counts_test[0] + class_counts_test[1])

This is an imbalanced dataset.

# **Exploratory Data Analysis**

## Univariate analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=True, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  

    #ax_box2.set_title(f"Boxplot of {feature}")  # Title for boxplot
    # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    ) 

In [ ]:
df_work = df.drop("Target", axis=1)
cols = df_work.columns.tolist()

for item in cols:
   histogram_boxplot(df, feature=item)


Most of these features have outliers. These will need to be addressed.

## Bivariate Analysis

In [ ]:
sns.pairplot(data=df_work);

In [ ]:
plt.figure(figsize=(10, 6), dpi=250)
plt.title("Correlation Between Features")
sns.heatmap(data=df_work.corr(), cmap="coolwarm", vmin=-1, vmax=1);

##### Find the 10 highest correlation pairs in the training dataset.

In [ ]:
corr_matrix = df.corr()
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs != 1]
sorted_corr_pairs = corr_pairs.abs().sort_values(ascending=False)
top_10_corr = sorted_corr_pairs.head(20)
print(top_10_corr)

Looking at both the Pair Plot and Heatmap of correlation coefficients there a few variable that correlate strongly but not enough to be concerned.

# **Data Preprocessing**

### Data Spliting

In [ ]:
# Splitting the dataset into Train, Validation and Test sets before applying data preprocessing techniques to help prevent data leakage.
y_test = df_test['Target']
X_test = df_test.drop("Target", axis=1)

In [ ]:
# First make an X and a y
y = df['Target']
X = df.drop("Target", axis=1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=random_seed)

### Dealing With Outliers
Every feature has a significant numbers of outliers, to prevent this from affecting the calculations we will apply a treatment.

In [ ]:
# Function to cap outliers at 3 standard deviations
def cap_outliers(df2, std_dev=3):
    for column in df2.select_dtypes(include=[np.number]).columns:
        mean = df2[column].mean()
        std = df2[column].std()
        
        # Calculate the upper and lower limits
        upper_limit = mean + std_dev * std
        lower_limit = mean - std_dev * std
        
        # Cap the outliers
        df2[column] = np.clip(df[column], lower_limit, upper_limit)
    
    return df2


In [ ]:
X_train = cap_outliers(X_train)
X_val = cap_outliers(X_val)
X_test = cap_outliers(X_test)

### Imputing Missing Values

In [ ]:
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train)
X_train = imputer.transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)

##### Verify that all missing values have been corrected. 

In [ ]:
missing_values = np.isnan(X_train).sum()
print(missing_values)

In [ ]:
missing_values = np.isnan(X_val).sum()
print(missing_values)

In [ ]:
missing_values = np.isnan(X_test).sum()
print(missing_values)

### Scaling the Data

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler.fit(X_train)

In [ ]:
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

### Another look at data after preprocessing
Applying a treatment on the outliers did not fix all of the outlier issues but it did reduce their affect. 

In [ ]:
df123 = pd.DataFrame(X_train)

cols = df123.columns.tolist()

for item in cols:
   histogram_boxplot(df123, feature=item)

# **Model Building**

### Utility Functions

In [ ]:
def plot(history, name):
    """
    Function to plot loss/accuracy

    history: an object which stores the metrics and losses.
    name: can be one of Loss or Accuracy
    """
    fig, ax = plt.subplots() #Creating a subplot with figure and axes.
    plt.plot(history.history[name]) #Plotting the train accuracy or train loss
    plt.plot(history.history['val_'+name]) #Plotting the validation accuracy or validation loss

    plt.title('Model ' + name.capitalize()) #Defining the title of the plot.
    plt.ylabel(name.capitalize()) #Capitalizing the first letter.
    plt.xlabel('Epoch') #Defining the label for the x-axis.
    fig.legend(['Train', 'Validation'], loc="outside right upper") #Defining the legend, loc controls the position of the legend.

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using statsmodels
def model_performance_classification(
    model, predictors, target, threshold=0.5
):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    threshold: threshold for classifying the observation as class 1
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors) > threshold
    # pred_temp = model.predict(predictors) > threshold
    # # rounding off the above values to get classes
    # pred = np.round(pred_temp)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred, average='weighted')  # to compute Recall
    precision = precision_score(target, pred, average='weighted')  # to compute Precision
    f1 = f1_score(target, pred, average='weighted')  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1 Score": f1,},
        index=[0],
    )

    return df_perf

## Model Evaluation Criterion

**Recall.** With "1" being a failure, positive and "0" being no failure we are most concerned with not identifying all of the failures. WIth no failure being the incorrect response. No failure but it failed, a false negative. Therefore Recall will be our primary metric. Since this dataset is so imbalanced we will also pay close attention to the F! Score but only for informational purposes.

### Model Setup Items

In [ ]:
# Calculate class weights for imbalanced dataset
cw = (y_train.shape[0]) / np.bincount(y_train)

# Create a dictionary mapping class indices to their respective class weights
cw_dict = {}
for i in range(cw.shape[0]):
    cw_dict[i] = cw[i]

cw_dict

In [ ]:
# defining the batch size and # epochs upfront as we'll be using the same values for all models
epochs = 50
batch_size = 100

## Initial Model Building (Model 0)

- Let's start with a neural network consisting of
  - just one hidden layer
  - activation function of ReLU
  - SGD as the optimizer

In [ ]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()

In [ ]:
#Initializing the neural network
model = Sequential()
model.add(Dense(25,activation="relu",input_dim=X_train.shape[1]))
model.add(Dense(1,activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
optimizer = tf.keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
start = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()

In [ ]:
print("Time taken in seconds ",end-start)

In [ ]:
plot(history,'loss')

In [ ]:
model_0_train_perf = model_performance_classification(model, X_train, y_train)
model_0_train_perf

In [ ]:
model_0_valid_perf = model_performance_classification(model, X_val, y_val)
model_0_valid_perf

Recall, our primary metric at 0.965 is a very good score. 

# **Model Performance Improvement**

## Model 1

On this model I am going to reduce the number of nodes in the hidden layer to 15.

In [ ]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()

In [ ]:
#Initializing the neural network
model = Sequential()
model.add(Dense(15,activation="relu",input_dim=X_train.shape[1]))
model.add(Dense(1,activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
optimizer = tf.keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
start = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()

In [ ]:
print("Time taken in seconds ",end-start)

In [ ]:
plot(history,'loss')

In [ ]:
model_1_train_perf = model_performance_classification(model, X_train, y_train)
model_1_train_perf

In [ ]:
model_1_valid_perf = model_performance_classification(model, X_val, y_val)
model_1_valid_perf

Decreasing the number of nodes in the hidden layer from 25 to 15 gave a small but marked decrease in performance. Will be returnin the number of nodes to a minimum of 25. I am going to increase them from 25 in the next model to see if there is an increase in perforamnce.

## Model 2

In this model I am going to increase the number of nodes in the hidden layer to 40 to see if there is an increase in poerformance.

In [ ]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()

In [ ]:
model = Sequential()
model.add(Dense(40,activation="relu",input_dim=X_train.shape[1]))
model.add(Dense(1,activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
optimizer = tf.keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
start = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()

In [ ]:
print("Time taken in seconds ",end-start)

In [ ]:
plot(history,'loss')

In [ ]:
model_2_train_perf = model_performance_classification(model, X_train, y_train)
model_2_train_perf

In [ ]:
model_2_valid_perf = model_performance_classification(model, X_val, y_val)
model_2_valid_perf

While slight, going to 40 nodes in the hidden layer provided an boost in performance. 40 Nodes in the hidden layer will now be the base. 

## Model 3

As we have are dealing with an imbalance in class distribution, we should also be using class weights to allow the model to give proportionally more importance to the minority class.

In [ ]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()

In [ ]:
model = Sequential()
model.add(Dense(40,activation="relu",input_dim=X_train.shape[1]))
model.add(Dense(1,activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
optimizer = tf.keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
start = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs,class_weight=cw_dict)
end=time.time()

## Model 4

Since we have used only SGD optimizer till now, let's use another kind of optimizer and observe its impact on the model performmance.

## Model 5

This time we will add more layers and dropout while using a different optimizer.

## Model 6

Let's see how does the model performance change when the model gives higher importance to the minority class by adding class weights along with dropout layer and different optimizer.

# **Model Performance Comparison and Final Model Selection**

Now, in order to select the final model, we will compare the performances of all the models for the training and validation sets.

Now, let's check the performance of the final model on the test set.

# **Actionable Insights and Recommendations**

Write down some insights and business recommendations based on your observations.